# ARC symbolic solver
Attach the competition data and disable Internet. This notebook embeds solver code and needs only the Python standard library. Local accuracy is reported in the project inspection report.

In [ ]:
from pathlib import Path
import sys, json
files = {'arc_solver.py': '"""Symbolic ARC program search and competition submission generation.\n\nThe solver only sees task demonstrations and test inputs. It does not read labels.\n"""\nfrom __future__ import annotations\nimport json\nfrom pathlib import Path\nfrom collections import Counter\n\nROOT = Path(__file__).resolve().parents[1]\nDATA = ROOT / "data"\nOUT = ROOT / "outputs" / "submission.json"\n\n\ndef key(g): return tuple(tuple(r) for r in g)\ndef copy(g): return [r[:] for r in g]\ndef dims(g): return len(g), len(g[0]) if g else 0\n\n\ndef transforms(g):\n    a=copy(g); out=[]\n    for _ in range(4):\n        out.append(a); a=[list(r) for r in zip(*a[::-1])]\n    out += [x[::-1] for x in out]\n    return out\n\n\ndef object_components(g, diagonal=False, background=None):\n    h,w=dims(g); seen=set(); objs=[]\n    for r in range(h):\n      for c in range(w):\n        color=g[r][c]\n        if (r,c) in seen or (background is not None and color==background): continue\n        stack=[(r,c)]; seen.add((r,c)); cells=[]\n        while stack:\n          y,x=stack.pop(); cells.append((y,x))\n          steps=[(1,0),(-1,0),(0,1),(0,-1)]\n          if diagonal: steps += [(1,1),(1,-1),(-1,1),(-1,-1)]\n          for dy,dx in steps:\n            yy,xx=y+dy,x+dx\n            if 0<=yy<h and 0<=xx<w and (yy,xx) not in seen and g[yy][xx]==color:\n              seen.add((yy,xx)); stack.append((yy,xx))\n        ys=[p[0] for p in cells]; xs=[p[1] for p in cells]\n        objs.append({\'color\':color,\'cells\':cells,\'box\':(min(ys),min(xs),max(ys)+1,max(xs)+1)})\n    return objs\n\n\ndef apply(inp, program):\n    h,w=dims(inp); op=program[0]\n    if op==\'compose\':\n      x=apply(inp,program[1])\n      return apply(x,program[2]) if x else None\n    if op==\'bbox\':\n      bg=program[1]\n      if bg==\'mode\': bg=Counter(v for row in inp for v in row).most_common(1)[0][0]\n      cells=[(r,c) for r in range(h) for c in range(w) if inp[r][c]!=bg]\n      if not cells: return None\n      y,x=min(r for r,c in cells),min(c for r,c in cells)\n      yy,xx=max(r for r,c in cells)+1,max(c for r,c in cells)+1\n      return [row[x:xx] for row in inp[y:yy]]\n    if op==\'object\':\n      bg,selection=program[1:]\n      if bg==\'mode\': bg=Counter(v for row in inp for v in row).most_common(1)[0][0]\n      objects=object_components(inp,background=bg)\n      if not objects: return None\n      ob=sorted(objects,key=lambda o:len(o[\'cells\']),reverse=selection==\'largest\')[0]\n      y,x,yy,xx=ob[\'box\']\n      return [row[x:xx] for row in inp[y:yy]]\n    if op==\'geom\':\n      return transforms(inp)[program[1]]\n    if op==\'color\':\n      m=program[1]; return [[m.get(v,v) for v in r] for r in inp]\n    if op==\'tile\':\n      sy,sx=program[1:]; oh,ow=h*sy,w*sx; return [[inp[r%h][c%w] for c in range(ow)] for r in range(oh)]\n    if op==\'scale\':\n      sy,sx=program[1:]; return [[v for x in r for v in [x]*sx] for r in inp for _ in range(sy)]\n    if op==\'crop\':\n      y,x,hh,ww=program[1:]\n      if y+hh>h or x+ww>w: return None\n      return [r[x:x+ww] for r in inp[y:y+hh]]\n    if op==\'majority\':\n      bg=program[1]; counts=Counter(v for r in inp for v in r if v!=bg)\n      fg=counts.most_common(1)[0][0] if counts else bg\n      return [[fg if v==bg else bg for v in r] for r in inp]\n    if op==\'fill_boxes\':\n      bg,fill=program[1:]; out=copy(inp)\n      for ob in object_components(inp,background=bg):\n        y1,x1,y2,x2=ob[\'box\']\n        if y2-y1>2 and x2-x1>2:\n          border={(y1,x) for x in range(x1,x2)}|{(y2-1,x) for x in range(x1,x2)}|{(y,x1) for y in range(y1,y2)}|{(y,x2-1) for y in range(y1,y2)}\n          if all(inp[y][x]==ob[\'color\'] for y,x in border):\n            for y in range(y1+1,y2-1):\n              for x in range(x1+1,x2-1): out[y][x]=fill\n      return out\n    if op==\'complete_lines\':\n      bg,fg=program[1:]; out=copy(inp)\n      for r in range(h):\n        if all(v in (bg,fg) for v in inp[r]) and fg in inp[r]: out[r]=[fg]*w\n      for c in range(w):\n        col=[inp[r][c] for r in range(h)]\n        if all(v in (bg,fg) for v in col) and fg in col:\n          for r in range(h): out[r][c]=fg\n      return out\n    return None\n\n\ndef candidates(inp, out):\n    ih,iw=dims(inp); oh,ow=dims(out); ps=[]\n    # Eight dihedral symmetries.\n    for i,x in enumerate(transforms(inp)):\n      if x==out: ps.append((\'geom\',i))\n    if ih and iw and oh%ih==0 and ow%iw==0:\n      ps.append((\'scale\',oh//ih,ow//iw))\n      ps.append((\'tile\',oh//ih,ow//iw))\n    if oh<=ih and ow<=iw:\n      for y in range(ih-oh+1):\n       for x in range(iw-ow+1):\n        if inp[y:y+oh] and [r[x:x+ow] for r in inp[y:y+oh]]==out: ps.append((\'crop\',y,x,oh,ow))\n    colors=set(v for r in inp for v in r)\n    if colors:\n      bg=min(colors,key=lambda c:sum(v==c for r in inp for v in r))\n      if (\'majority\',bg) not in ps and apply(inp,(\'majority\',bg))==out: ps.append((\'majority\',bg))\n    for bg in colors:\n      for fill in range(10):\n        p=(\'fill_boxes\',bg,fill)\n        if apply(inp,p)==out: ps.append(p)\n      for fg in colors-{bg}:\n        p=(\'complete_lines\',bg,fg)\n        if apply(inp,p)==out: ps.append(p)\n    return [p for p in ps if apply(inp,p)==out]\n\n\ndef infer(train):\n    if not train: return []\n    programs=candidates(train[0][\'input\'],train[0][\'output\'])\n    bases=[(\'geom\',i) for i in range(8)]\n    for bg in [\'mode\']+list(range(10)):\n        bases.append((\'bbox\',bg))\n        bases.extend((\'object\',bg,which) for which in (\'largest\',\'smallest\'))\n    # Infer mapping jointly from all demonstrations, including many-to-one maps.\n    for base in bases:\n        transformed=[apply(ex[\'input\'],base) for ex in train]\n        if any(x is None or dims(x)!=dims(ex[\'output\']) for x,ex in zip(transformed,train)):\n            continue\n        if all(x==ex[\'output\'] for x,ex in zip(transformed,train)):\n            programs.append(base)\n            continue\n        mapping={}; valid=True\n        for x,ex in zip(transformed,train):\n            for row,target in zip(x,ex[\'output\']):\n                for a,b in zip(row,target):\n                    if a in mapping and mapping[a]!=b: valid=False\n                    mapping[a]=b\n        if valid: programs.append((\'compose\',base,(\'color\',mapping)))\n    return [p for p in programs if all(apply(ex[\'input\'],p)==ex[\'output\'] for ex in train)]\n\n\ndef predict_task(task, attempts=2):\n    if type(attempts) is not int or attempts != 2:\n        raise ValueError(\'ARC submissions require exactly two attempts\')\n    programs=infer(task.get(\'train\',[])); result=[]\n    for test in task.get(\'test\',[]):\n      preds=[]\n      for p in programs:\n        y=apply(test[\'input\'],p)\n        if valid_grid(y) and y not in preds: preds.append(y)\n      if not preds:\n        # Explicitly low-information fallback, with distinct shapes if available.\n        for ex in task.get(\'train\',[]):\n          if ex[\'output\'] not in preds: preds.append(copy(ex[\'output\']))\n      if not preds: preds=[copy(test[\'input\'])]\n      while len(preds)<attempts: preds.append(copy(preds[-1]))\n      result.append({f\'attempt_{i+1}\':preds[i] for i in range(attempts)})\n    return result\n\n\ndef valid_grid(g):\n    return isinstance(g,list) and 1<=len(g)<=30 and isinstance(g[0],list) and 1<=len(g[0])<=30 and all(isinstance(row,list) and len(row)==len(g[0]) and all(type(v) is int and 0<=v<=9 for v in row) for row in g)\n\n\ndef generate(challenges_path=DATA/\'arc-agi_test_challenges.json\',out_path=OUT):\n    challenges=json.loads(Path(challenges_path).read_text())\n    result={tid:predict_task(task) for tid,task in challenges.items()}\n    from validate import validate_submission\n    validate_submission(challenges,result)\n    Path(out_path).parent.mkdir(parents=True,exist_ok=True)\n    Path(out_path).write_text(json.dumps(result,separators=(\',\',\':\')))\n    return result\n\nif __name__==\'__main__\':\n    import argparse\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--challenges\',type=Path,default=DATA/\'arc-agi_test_challenges.json\')\n    parser.add_argument(\'--output\',type=Path,default=OUT)\n    args=parser.parse_args()\n    generate(args.challenges,args.output); print(f\'Wrote {args.output}\')\n', 'validate.py': '"""Strict submission validation against challenge inputs, without answer access."""\nimport json\nimport argparse\n\ndef validate_submission(challenges, submission):\n    from arc_solver import valid_grid\n    if not isinstance(submission,dict) or set(submission)!=set(challenges):\n        raise ValueError(\'Submission task IDs must exactly match challenges\')\n    total=0\n    for tid,task in challenges.items():\n        records=submission[tid]\n        if not isinstance(records,list) or len(records)!=len(task[\'test\']):\n            raise ValueError(f\'{tid}: incorrect number of test outputs\')\n        for i,record in enumerate(records):\n            if not isinstance(record,dict) or set(record)!={\'attempt_1\',\'attempt_2\'}:\n                raise ValueError(f\'{tid}/{i}: exactly two attempts required\')\n            for name,grid in record.items():\n                if not valid_grid(grid): raise ValueError(f\'{tid}/{i}/{name}: invalid grid\')\n            total+=1\n    return {\'tasks\':len(challenges),\'test_outputs\':total,\'attempts\':2*total}\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(); p.add_argument(\'challenges\');p.add_argument(\'submission\');a=p.parse_args()\n    print(validate_submission(json.load(open(a.challenges)),json.load(open(a.submission))))\n'}
code_dir=Path('/kaggle/working/arc_code')
code_dir.mkdir(exist_ok=True)
for name,text in files.items():
    (code_dir/name).write_text(text)
sys.path.insert(0,str(code_dir))
from arc_solver import generate
from validate import validate_submission
paths=list(Path('/kaggle/input').rglob('arc-agi_test_challenges.json'))
if len(paths)!=1:
    raise RuntimeError(f'Expected one competition challenge file; found {len(paths)}')
challenges=json.loads(paths[0].read_text())
predictions=generate(paths[0],Path('/kaggle/working/submission.json'))
print(validate_submission(challenges,predictions))
